# GulfDealFlow — Script 02b: News Staging Cleaner

Takes the raw `news_staging.csv` and:
- Removes market summary articles (not individual deals)
- Cleans messy company names
- Rejects absurd amounts (SWF totals, market aggregates)
- Extracts company names from headlines when blank
- Outputs a clean `news_staging_clean.csv` ready for Script 03

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install pandas -q
print('✓ Ready')

In [ ]:
import pandas as pd
import re
import os

BASE_DIR     = '/content/drive/MyDrive/GulfDealFlow'
INPUT_PATH   = os.path.join(BASE_DIR, 'news_staging.csv')
OUTPUT_PATH  = os.path.join(BASE_DIR, 'news_staging_clean.csv')
REMOVED_PATH = os.path.join(BASE_DIR, 'news_staging_removed.csv')  # so you can audit what was dropped

df = pd.read_csv(INPUT_PATH, dtype=str).fillna('')
print(f'Loaded {len(df)} rows from news_staging.csv')

In [ ]:
# ── STEP 1: Remove market summaries and non-deal articles ────────
# These are headlines about the ecosystem, not individual company deals

NON_DEAL_PATTERNS = [
    r'leads? mena',
    r'leads? gcc',
    r'startup funding in (?:q[1-4]|h[12]|january|february|march|april|may|june|july|august|september|october|november|december)',
    r'\d+ deals?',
    r'funding (?:rebounds?|climbs?|surges?|reaches?|hits?)',
    r'mena startup funding',
    r'vc market',
    r'investment report',
    r'ecosystem',
    r'seeks to increase',
    r'number of startups',
    r'conference',
    r'launches? .{0,20}fund',          # fund launches (not startup raises)
    r'fund of funds',
    r'sovereign wealth',
    r'\$[0-9]+\s*billion.{0,30}fund',  # billion-dollar funds
    r'invests? \$[0-9]+ (?:million|billion) in (?:startups|the region|international)',
    r'startup wrap',
    r'mena digest',
    r'weekly roundup',
    r'struggles? to attract',
    r'dominate mena',
    r'plots? a \$',
    r'pipeline to vcs',
    r'celebrates? emirati',
    r'smarter paths for',
    r'fundraising in \d{4}',
    r'future of entrepreneurship',
]

def is_non_deal(row):
    notes = row.get('notes', '').lower()
    for pat in NON_DEAL_PATTERNS:
        if re.search(pat, notes):
            return True
    return False

mask_non_deal = df.apply(is_non_deal, axis=1)
removed = df[mask_non_deal].copy()
df = df[~mask_non_deal].copy()
print(f'Removed {len(removed)} non-deal rows (market summaries, fund launches, reports)')
print(f'Remaining: {len(df)} rows')

In [ ]:
# ── STEP 2: Remove absurd amounts ───────────────────────────────
# Individual startup deals are almost never above $500M
# Anything higher is likely a market aggregate or SWF figure

MAX_DEAL_SIZE = 500_000_000  # $500M hard cap

def amount_is_absurd(row):
    amt = row.get('amount_usd', '')
    if amt == '' or amt == '0':
        return False
    try:
        return float(amt) > MAX_DEAL_SIZE
    except:
        return False

mask_absurd = df.apply(amount_is_absurd, axis=1)
removed = pd.concat([removed, df[mask_absurd]])
df = df[~mask_absurd].copy()
print(f'Removed {mask_absurd.sum()} rows with absurd amounts (>{MAX_DEAL_SIZE/1e6:.0f}M — likely aggregates)')
print(f'Remaining: {len(df)} rows')

In [ ]:
# ── STEP 3: Clean company names ──────────────────────────────────

# Prefixes to strip from auto-extracted names
NOISE_PREFIXES = [
    r'^uae[\-\s](?:based|blockchain|biotech|startup|fintech|tech|ai|web3)?\s*(?:startup\s*)?',
    r'^dubai[\-\s](?:based|startup|based startup)?\s*',
    r'^saudi[\-\s](?:based|startup|arabia[\-\s]based)?\s*',
    r'^riyadh[\-\s](?:based)?\s*',
    r'^abu dhabi[\-\s](?:based)?\s*',
    r'^qatari[\-\s](?:fintech|startup)?\s*',
    r'^bahraini[\-\s](?:startup)?\s*',
    r'^gcc[\-\s](?:based|expansion)?[\-\s]',
    r'^mena[\-\s]',
    r'^this\s+',
    r"^uae'?s?\s+",
    r"^dubai'?s?\s+",
    r"^saudi'?s?\s+",
]

# Words that mean the extracted name is not a company
INVALID_NAMES = {
    'uae', 'dubai', 'saudi', 'saudi arabia', 'gcc', 'mena', 'the', 'a', 'an',
    'february', 'january', 'march', 'april', 'may', 'june', 'july',
    'august', 'september', 'october', 'november', 'december',
    'us ai startup', 'startup', 'fintech', 'healthtech'
}

def clean_company_name(row):
    name = row.get('company_name', '').strip()

    # If blank, try extracting from the notes headline
    if not name:
        notes = row.get('notes', '')
        # Extract text between 'REVIEW | ' and ' | http'
        m = re.search(r'REVIEW \| (.+?) \| http', notes)
        headline = m.group(1) if m else ''

        # Try common headline patterns
        patterns = [
            r"^([A-Z][a-zA-Z0-9\s\-\.&']+?)\s+(?:raises?|secures?|closes?|lands?|bags?|banks?)\s",
            r"(?:startup|fintech|proptech|healthtech)\s+([A-Z][a-zA-Z0-9\-]+)\s+(?:raises?|secures?)",
        ]
        for pat in patterns:
            m = re.search(pat, headline)
            if m:
                name = m.group(1).strip()
                break

    if not name:
        return ''

    # Strip geographic/descriptive prefixes
    for prefix in NOISE_PREFIXES:
        name = re.sub(prefix, '', name, flags=re.IGNORECASE).strip()

    # Reject if name is a known non-company word
    if name.lower() in INVALID_NAMES:
        return ''

    # Reject if too long (probably a sentence fragment)
    if len(name) > 35 or name.count(' ') > 4:
        return ''

    return name

df['company_name'] = df.apply(clean_company_name, axis=1)

blank_after = (df['company_name'] == '').sum()
print(f'Company names cleaned')
print(f'Still blank after cleaning: {blank_after} rows (will need manual review)')

In [ ]:
# ── STEP 4: Fix stage mismatches ────────────────────────────────
# Some stages got miscategorised (e.g. Series A used for Pre-Series A)

def fix_stage(row):
    stage = row.get('stage', '')
    notes = row.get('notes', '').lower()

    # Pre-Series A should be Seed not Series A
    if stage == 'Series A' and 'pre-series a' in notes:
        return 'Seed'
    # Pre-Seed check
    if 'pre-seed' in notes or 'pre seed' in notes:
        return 'Pre-Seed'
    return stage

df['stage'] = df.apply(fix_stage, axis=1)
print('Stages fixed')

In [ ]:
# ── STEP 5: Fix sectors from headline ───────────────────────────
# Re-run sector detection on the full headline since 'Other' is overrepresented

SECTOR_KEYWORDS = {
    'Fintech':                   ['fintech', 'payment', 'neobank', 'lending', 'insurtech', 'crypto', 'remittance', 'bnpl', 'buy now pay later', 'insurance claim', 'banking'],
    'Proptech':                  ['proptech', 'real estate', 'property', 'mortgage', 'rental', 'rent'],
    'Logistics & Supply Chain':  ['logistics', 'supply chain', 'last mile', 'freight', 'delivery', 'fleet', 'trucking', 'shipping'],
    'Healthtech':                ['healthtech', 'telehealth', 'medtech', 'digital health', 'pharma', 'cancer', 'fertility', 'biotech', 'medical'],
    'Edtech':                    ['edtech', 'education', 'learning', 'school', 'university', 'k-12'],
    'E-commerce & Retail':       ['e-commerce', 'ecommerce', 'marketplace', 'retail', 'd2c', 'gifting', 'cleaning', 'on-demand'],
    'SaaS & Enterprise Software':['saas', 'enterprise software', 'b2b software', 'cloud', 'procurement', 'construction tech', 'point-of-sale', 'pos'],
    'Deep Tech & AI':            ['ai', 'artificial intelligence', 'machine learning', 'deep tech', 'robotics', 'blockchain', 'web3', 'data'],
    'Energy & Cleantech':        ['cleantech', 'clean energy', 'solar', 'renewable', 'green', 'climate', 'agri'],
    'Media & Entertainment':     ['media', 'streaming', 'gaming', 'content', 'creator'],
    'Food & Agritech':           ['food', 'restaurant', 'cloud kitchen', 'meal', 'kitchen', 'agritech', 'harvest'],
}

def reclassify_sector(row):
    current = row.get('sector', 'Other')
    if current != 'Other':
        return current  # keep if already classified
    notes = row.get('notes', '').lower()
    for sector, keywords in SECTOR_KEYWORDS.items():
        for kw in keywords:
            if kw in notes:
                return sector
    return 'Other'

df['sector'] = df.apply(reclassify_sector, axis=1)
print('Sectors reclassified')
print(df['sector'].value_counts().to_string())

In [ ]:
# ── STEP 6: Save outputs ─────────────────────────────────────────

df.to_csv(OUTPUT_PATH, index=False)
removed.to_csv(REMOVED_PATH, index=False)

print(f'\n{"="*50}')
print(f'CLEANING COMPLETE')
print(f'{"="*50}')
print(f'Clean deals saved:   {len(df)} rows → news_staging_clean.csv')
print(f'Removed rows saved:  {len(removed)} rows → news_staging_removed.csv')
print(f'\nBlank company names still needing review: {(df["company_name"]=="").sum()}')
print(f'\nNext steps:')
print(f'  1. Open news_staging_clean.csv in Drive')
print(f'  2. Fill in blank company names using the notes column')
print(f'  3. Add descriptions, websites, investor info where you can')
print(f'  4. Update INPUT_PATH in Script 03 to use news_staging_clean.csv')
print(f'  5. Run Script 03 to merge into master')

In [ ]:
# Preview clean output
df[['company_name','country','city','date','stage','amount_usd','sector']].head(30)